In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [2]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_15_18.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 변수 추가

In [3]:
df11=pd.read_parquet(r'data/train/1.회원정보/201807_train_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df12=pd.read_parquet(r'data/train/1.회원정보/201808_train_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df13=pd.read_parquet(r'data/train/1.회원정보/201809_train_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df14=pd.read_parquet(r'data/train/1.회원정보/201810_train_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df15=pd.read_parquet(r'data/train/1.회원정보/201811_train_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df16=pd.read_parquet(r'data/train/1.회원정보/201812_train_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])

In [4]:
df_test11=pd.read_parquet(r'data/test/1.회원정보/201807_test_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df_test12=pd.read_parquet(r'data/test/1.회원정보/201808_test_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df_test13=pd.read_parquet(r'data/test/1.회원정보/201809_test_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df_test14=pd.read_parquet(r'data/test/1.회원정보/201810_test_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df_test15=pd.read_parquet(r'data/test/1.회원정보/201811_test_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])
df_test16=pd.read_parquet(r'data/test/1.회원정보/201812_test_.parquet',columns=['ID','기준년월','입회경과개월수_신용','_1순위카드이용건수'])

In [5]:
df21=pd.read_parquet(r'data/train/2.신용정보/201807_train_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df22=pd.read_parquet(r'data/train/2.신용정보/201808_train_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df23=pd.read_parquet(r'data/train/2.신용정보/201809_train_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df24=pd.read_parquet(r'data/train/2.신용정보/201810_train_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df25=pd.read_parquet(r'data/train/2.신용정보/201811_train_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df26=pd.read_parquet(r'data/train/2.신용정보/201812_train_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])

In [6]:
df_test21=pd.read_parquet(r'data/test/2.신용정보/201807_test_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df_test22=pd.read_parquet(r'data/test/2.신용정보/201808_test_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df_test23=pd.read_parquet(r'data/test/2.신용정보/201809_test_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df_test24=pd.read_parquet(r'data/test/2.신용정보/201810_test_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df_test25=pd.read_parquet(r'data/test/2.신용정보/201811_test_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])
df_test26=pd.read_parquet(r'data/test/2.신용정보/201812_test_.parquet',columns=['ID','기준년월','CA한도금액','카드이용한도금액' ,'카드이용한도금액_B1M','카드이용한도금액_B2M'])

In [7]:
df31=pd.read_parquet(r'data/train/3.승인매출정보/201807_train_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df32=pd.read_parquet(r'data/train/3.승인매출정보/201808_train_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df33=pd.read_parquet(r'data/train/3.승인매출정보/201809_train_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df34=pd.read_parquet(r'data/train/3.승인매출정보/201810_train_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df35=pd.read_parquet(r'data/train/3.승인매출정보/201811_train_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df36=pd.read_parquet(r'data/train/3.승인매출정보/201812_train_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])

In [8]:
df_test31=pd.read_parquet(r'data/test/3.승인매출정보/201807_test_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df_test32=pd.read_parquet(r'data/test/3.승인매출정보/201808_test_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df_test33=pd.read_parquet(r'data/test/3.승인매출정보/201809_test_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df_test34=pd.read_parquet(r'data/test/3.승인매출정보/201810_test_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df_test35=pd.read_parquet(r'data/test/3.승인매출정보/201811_test_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])
df_test36=pd.read_parquet(r'data/test/3.승인매출정보/201812_test_.parquet',columns=['ID','기준년월','이용후경과월_할부_무이자' ,'이용개월수_할부_R12M','이용개월수_할부_무이자_R6M'])

In [15]:
df71=pd.read_parquet(r'data/train/7.마케팅정보/201807_train_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df72=pd.read_parquet(r'data/train/7.마케팅정보/201807_train_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df73=pd.read_parquet(r'data/train/7.마케팅정보/201807_train_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df74=pd.read_parquet(r'data/train/7.마케팅정보/201807_train_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df75=pd.read_parquet(r'data/train/7.마케팅정보/201807_train_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df76=pd.read_parquet(r'data/train/7.마케팅정보/201807_train_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])

In [16]:
df_test71=pd.read_parquet(r'data/test/7.마케팅정보/201807_test_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df_test72=pd.read_parquet(r'data/test/7.마케팅정보/201808_test_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df_test73=pd.read_parquet(r'data/test/7.마케팅정보/201809_test_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df_test74=pd.read_parquet(r'data/test/7.마케팅정보/201810_test_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df_test75=pd.read_parquet(r'data/test/7.마케팅정보/201811_test_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])
df_test76=pd.read_parquet(r'data/test/7.마케팅정보/201812_test_.parquet',columns=['ID','기준년월','캠페인접촉건수_R12M'])

In [11]:
all_df1 = pd.concat([df11, df12, df13, df14, df15, df16])
all_df1.reset_index(inplace=True, drop=True)
all_df1

,ID,기준년월,입회경과개월수_신용,_1순위카드이용건수
0,TRAIN_000000,201807,67,26
1,TRAIN_000001,201807,12,46
2,TRAIN_000002,201807,124,28
3,TRAIN_000003,201807,27,1
4,TRAIN_000004,201807,2,-2
...,...,...,...,...
2399995,TRAIN_399995,201812,209,3
2399996,TRAIN_399996,201812,17,38
2399997,TRAIN_399997,201812,115,33
2399998,TRAIN_399998,201812,71,-2


In [12]:
all_df2 = pd.concat([df21, df22, df23, df24, df25, df26])
all_df2.reset_index(inplace=True, drop=True)
all_df2

,ID,기준년월,CA한도금액,카드이용한도금액,카드이용한도금액_B1M,카드이용한도금액_B2M
0,TRAIN_000000,201807,7270,19354,20805,19723
1,TRAIN_000001,201807,5718,9996,10000,9998
2,TRAIN_000002,201807,35207,88193,78730,77975
3,TRAIN_000003,201807,6531,19062,20523,19226
4,TRAIN_000004,201807,47149,177222,169667,168681
...,...,...,...,...,...,...
2399995,TRAIN_399995,201812,10167,20070,21097,21152
2399996,TRAIN_399996,201812,31159,84217,78997,78140
2399997,TRAIN_399997,201812,19429,52612,61315,63374
2399998,TRAIN_399998,201812,4228,10002,10002,10001


In [13]:
all_df3 = pd.concat([df31, df32, df33, df34, df35, df36])
all_df3.reset_index(inplace=True, drop=True)
all_df3

,ID,기준년월,이용후경과월_할부_무이자,이용개월수_할부_R12M,이용개월수_할부_무이자_R6M
0,TRAIN_000000,201807,2,5,2
1,TRAIN_000001,201807,9,2,0
2,TRAIN_000002,201807,0,2,2
3,TRAIN_000003,201807,1,8,3
4,TRAIN_000004,201807,12,0,0
...,...,...,...,...,...
2399995,TRAIN_399995,201812,12,0,0
2399996,TRAIN_399996,201812,9,1,0
2399997,TRAIN_399997,201812,1,4,2
2399998,TRAIN_399998,201812,12,0,0


In [17]:
all_df7 = pd.concat([df71, df72, df73, df74, df75, df76])
all_df7.reset_index(inplace=True, drop=True)
all_df7

,ID,기준년월,캠페인접촉건수_R12M
0,TRAIN_000000,201807,1회 이상
1,TRAIN_000001,201807,15회 이상
2,TRAIN_000002,201807,1회 이상
3,TRAIN_000003,201807,1회 이상
4,TRAIN_000004,201807,1회 이상
...,...,...,...
2399995,TRAIN_399995,201807,1회 이상
2399996,TRAIN_399996,201807,15회 이상
2399997,TRAIN_399997,201807,1회 이상
2399998,TRAIN_399998,201807,1회 이상


In [18]:
all_testdf1 = pd.concat([df_test11, df_test12, df_test13, df_test14, df_test15, df_test16])
all_testdf1.reset_index(inplace=True, drop=True)
all_testdf1

,ID,기준년월,입회경과개월수_신용,_1순위카드이용건수
0,TEST_00000,201807,51,51
1,TEST_00001,201807,30,40
2,TEST_00002,201807,5,154
3,TEST_00003,201807,73,105
4,TEST_00004,201807,176,52
...,...,...,...,...
599995,TEST_99995,201812,69,-2
599996,TEST_99996,201812,14,4
599997,TEST_99997,201812,6,6
599998,TEST_99998,201812,82,185


In [19]:
all_testdf2 = pd.concat([df_test21, df_test22, df_test23, df_test24, df_test25, df_test26])
all_testdf2.reset_index(inplace=True, drop=True)
all_testdf2

,ID,기준년월,CA한도금액,카드이용한도금액,카드이용한도금액_B1M,카드이용한도금액_B2M
0,TEST_00000,201807,18131,50902,49999,50006
1,TEST_00001,201807,16819,50080,50000,50003
2,TEST_00002,201807,30505,100045,100053,100056
3,TEST_00003,201807,6402,18508,21035,19693
4,TEST_00004,201807,0,4033,4291,3924
...,...,...,...,...,...,...
599995,TEST_99995,201812,0,0,0,0
599996,TEST_99996,201812,17876,49025,50008,49990
599997,TEST_99997,201812,13332,29996,30011,30004
599998,TEST_99998,201812,17362,42610,37999,37139


In [20]:
all_testdf3 = pd.concat([df_test31, df_test32, df_test33, df_test34, df_test35, df_test36])
all_testdf3.reset_index(inplace=True, drop=True)
all_testdf3

,ID,기준년월,이용후경과월_할부_무이자,이용개월수_할부_R12M,이용개월수_할부_무이자_R6M
0,TEST_00000,201807,12,0,0
1,TEST_00001,201807,0,10,6
2,TEST_00002,201807,3,7,1
3,TEST_00003,201807,2,2,1
4,TEST_00004,201807,0,7,4
...,...,...,...,...,...
599995,TEST_99995,201812,12,0,0
599996,TEST_99996,201812,12,0,0
599997,TEST_99997,201812,12,0,0
599998,TEST_99998,201812,2,3,1


In [21]:
all_testdf7 = pd.concat([df_test71, df_test72, df_test73, df_test74, df_test75, df_test76])
all_testdf7.reset_index(inplace=True, drop=True)
all_testdf7

,ID,기준년월,캠페인접촉건수_R12M
0,TEST_00000,201807,1회 이상
1,TEST_00001,201807,5회 이상
2,TEST_00002,201807,10회 이상
3,TEST_00003,201807,15회 이상
4,TEST_00004,201807,1회 이상
...,...,...,...
599995,TEST_99995,201812,1회 이상
599996,TEST_99996,201812,1회 이상
599997,TEST_99997,201812,1회 이상
599998,TEST_99998,201812,5회 이상


### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [22]:
# 데이터를 읽어온다.
train_df = pd.read_parquet('Seleted(C_D)_delecteABE_all_train.parquet')
test_df = pd.read_parquet('Seleted(C_D)_delecteABE_all_test.parquet')

display(train_df)
display(test_df)

,기준년월,ID,Segment,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,...,청구금액_R3M,청구금액_B0,청구서발송여부_B0,할인건수_R3M,할인건수_B0M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,46588,12226,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,10회 이상
1,201807,TRAIN_000002,C,23988,24493,23988,1,자녀출산기,9,9,...,85931,21866,1,1회 이상,1회 이상,30회 이상,10회 이상,10회 이상,1회 이상,1회 이상
2,201807,TRAIN_000003,D,3904,5933,3904,1,자녀성장(2),12,12,...,61518,16356,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,10회 이상
3,201807,TRAIN_000008,C,124967,68078,121279,5,자녀출산기,12,12,...,62715,20512,1,10회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TRAIN_000010,D,21001,18796,21001,1,자녀성장(1),12,12,...,30449,22512,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476827,201812,TRAIN_399979,D,31187,27337,31187,2,자녀성장(2),12,12,...,41812,11817,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
476828,201812,TRAIN_399987,C,42492,35751,42492,1,자녀성장(2),12,12,...,68356,17859,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,30회 이상
476829,201812,TRAIN_399993,C,72348,27792,72348,4,자녀성장(1),12,12,...,34890,10810,1,20회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상
476830,201812,TRAIN_399996,D,27636,26357,27636,1,자녀성장(2),12,12,...,37515,14402,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상


,기준년월,ID,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,이용개월수_일시불_R12M,...,청구금액_R3M,청구금액_B0,청구서발송여부_B0,할인건수_R3M,할인건수_B0M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,201807,TEST_00000,21458,13852,21458,2,자녀성장(1),10,10,10,...,11441,4931,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
1,201807,TEST_00001,18681,11065,10759,2,자녀독립기,9,9,9,...,20522,10152,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
2,201807,TEST_00002,40758,27071,40758,2,자녀성장(1),12,12,12,...,50508,13223,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
3,201807,TEST_00003,5255,4827,5255,1,자녀성장(1),9,9,9,...,4604,2112,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TEST_00004,16148,8011,14290,3,자녀성장(1),12,12,12,...,6788,4406,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,201812,TEST_99995,0,0,0,0,노년생활,0,0,0,...,0,0,0,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
599996,201812,TEST_99996,3110,1231,3110,1,자녀출산기,10,9,9,...,1256,359,1,1회 이상,1회 이상,10회 이상,1회 이상,1회 이상,1회 이상,1회 이상
599997,201812,TEST_99997,0,0,0,0,자녀성장(1),0,0,0,...,0,0,0,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
599998,201812,TEST_99998,173263,63592,113786,6,가족구축기,12,12,12,...,48141,21273,1,1회 이상,1회 이상,40회 이상,1회 이상,1회 이상,1회 이상,1회 이상


In [23]:
# ID와 기준년월 기준으로 join (왼쪽 기준으로 추가)
train_df = train_df.merge(all_df1, on=['ID', '기준년월'], how='left')
train_df = train_df.merge(all_df2, on=['ID', '기준년월'], how='left')
train_df = train_df.merge(all_df3, on=['ID', '기준년월'], how='left')
train_df = train_df.merge(all_df7, on=['ID', '기준년월'], how='left')

In [24]:
train_df

,기준년월,ID,Segment,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,...,입회경과개월수_신용,_1순위카드이용건수,CA한도금액,카드이용한도금액,카드이용한도금액_B1M,카드이용한도금액_B2M,이용후경과월_할부_무이자,이용개월수_할부_R12M,이용개월수_할부_무이자_R6M,캠페인접촉건수_R12M
0,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
1,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
2,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
3,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
4,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
874187,201812,TRAIN_399979,D,31187,27337,31187,2,자녀성장(2),12,12,...,248,52,26268,88823,76798,84977,1,8,4,NaN
874188,201812,TRAIN_399987,C,42492,35751,42492,1,자녀성장(2),12,12,...,113,49,10026,42192,37302,37740,1,10,4,NaN
874189,201812,TRAIN_399993,C,72348,27792,72348,4,자녀성장(1),12,12,...,127,177,22513,53999,60057,61134,12,0,0,NaN
874190,201812,TRAIN_399996,D,27636,26357,27636,1,자녀성장(2),12,12,...,17,38,31159,84217,78997,78140,9,1,0,NaN


In [25]:
# ID와 기준년월 기준으로 join (왼쪽 기준으로 추가)
test_df = test_df.merge(all_testdf1, on=['ID', '기준년월'], how='left')
test_df = test_df.merge(all_testdf2, on=['ID', '기준년월'], how='left')
test_df = test_df.merge(all_testdf3, on=['ID', '기준년월'], how='left')
test_df = test_df.merge(all_testdf7, on=['ID', '기준년월'], how='left')

In [26]:
test_df

,기준년월,ID,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,이용개월수_일시불_R12M,...,입회경과개월수_신용,_1순위카드이용건수,CA한도금액,카드이용한도금액,카드이용한도금액_B1M,카드이용한도금액_B2M,이용후경과월_할부_무이자,이용개월수_할부_R12M,이용개월수_할부_무이자_R6M,캠페인접촉건수_R12M
0,201807,TEST_00000,21458,13852,21458,2,자녀성장(1),10,10,10,...,51,51,18131,50902,49999,50006,12,0,0,1회 이상
1,201807,TEST_00001,18681,11065,10759,2,자녀독립기,9,9,9,...,30,40,16819,50080,50000,50003,0,10,6,5회 이상
2,201807,TEST_00002,40758,27071,40758,2,자녀성장(1),12,12,12,...,5,154,30505,100045,100053,100056,3,7,1,10회 이상
3,201807,TEST_00003,5255,4827,5255,1,자녀성장(1),9,9,9,...,73,105,6402,18508,21035,19693,2,2,1,15회 이상
4,201807,TEST_00004,16148,8011,14290,3,자녀성장(1),12,12,12,...,176,52,0,4033,4291,3924,0,7,4,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,201812,TEST_99995,0,0,0,0,노년생활,0,0,0,...,69,-2,0,0,0,0,12,0,0,1회 이상
599996,201812,TEST_99996,3110,1231,3110,1,자녀출산기,10,9,9,...,14,4,17876,49025,50008,49990,12,0,0,1회 이상
599997,201812,TEST_99997,0,0,0,0,자녀성장(1),0,0,0,...,6,6,13332,29996,30011,30004,12,0,0,1회 이상
599998,201812,TEST_99998,173263,63592,113786,6,가족구축기,12,12,12,...,82,185,17362,42610,37999,37139,2,3,1,5회 이상


In [27]:
notE = pd.read_csv('data/model3_notE(2).csv')
notE

,ID,Segment
0,TEST_00010,Not_E
1,TEST_00016,Not_E
2,TEST_00032,Not_E
3,TEST_00034,Not_E
4,TEST_00037,Not_E
...,...,...
16698,TEST_99947,Not_E
16699,TEST_99957,Not_E
16700,TEST_99961,Not_E
16701,TEST_99982,Not_E


In [29]:
# 1. 'Not_E'인 ID만 필터링
filtered_ids1 = notE[notE['Segment'] == 'Not_E']['ID']

# 2. df2에서 해당 ID만 골라오기
df2 = test_df[test_df['ID'].isin(filtered_ids1)]

In [30]:
notAB = pd.read_csv('data/model3_notAB(4).csv')
notAB

,ID,Segment
0,TEST_00010,notAB
1,TEST_00016,notAB
2,TEST_00032,notAB
3,TEST_00034,notAB
4,TEST_00037,notAB
...,...,...
16684,TEST_99947,notAB
16685,TEST_99957,notAB
16686,TEST_99961,notAB
16687,TEST_99982,notAB


In [31]:
# 1. 'Not_E'인 ID만 필터링
filtered_ids2 = notAB[notAB['Segment'] == 'notAB']['ID']

# 2. df2에서 해당 ID만 골라오기
test_df2 = df2[df2['ID'].isin(filtered_ids2)]

In [64]:
test_df2.to_csv('C,D(18).csv', index=False, encoding='utf-8-sig')

In [33]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df2])
all_df.reset_index(inplace=True, drop=True)
all_df

,기준년월,ID,Segment,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,...,입회경과개월수_신용,_1순위카드이용건수,CA한도금액,카드이용한도금액,카드이용한도금액_B1M,카드이용한도금액_B2M,이용후경과월_할부_무이자,이용개월수_할부_R12M,이용개월수_할부_무이자_R6M,캠페인접촉건수_R12M
0,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
1,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
2,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
3,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
4,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,67,26,7270,19354,20805,19723,2,5,2,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974321,201812,TEST_99947,NaN,87532,41465,87532,3,자녀성장(1),12,12,...,93,122,49064,180204,170672,194680,6,2,0,10회 이상
974322,201812,TEST_99957,NaN,61148,38644,59950,4,자녀성장(1),12,12,...,17,115,19032,52373,50015,50010,4,2,1,1회 이상
974323,201812,TEST_99961,NaN,30318,25961,30318,1,자녀성장(2),12,12,...,120,57,18298,45661,36138,37379,12,0,0,1회 이상
974324,201812,TEST_99982,NaN,94440,70955,94440,1,노년생활,12,12,...,92,101,58290,199981,197044,197112,12,0,0,1회 이상


In [34]:
all_df.drop(columns=['Segment','ID','기준년월'], inplace=True)

In [35]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 974326 entries, 0 to 974325
Data columns (total 69 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   이용금액_R3M_신용체크       974326 non-null  int64 
 1   _1순위카드이용금액          974326 non-null  int64 
 2   이용금액_R3M_신용         974326 non-null  int64 
 3   이용카드수_신용체크          974326 non-null  int64 
 4   Life_Stage          974326 non-null  object
 5   이용개월수_신용_R12M       974326 non-null  int64 
 6   이용개월수_신판_R12M       974326 non-null  int64 
 7   이용개월수_일시불_R12M      974326 non-null  int64 
 8   이용금액_일시불_R6M        974326 non-null  int64 
 9   이용금액_일시불_B0M        974326 non-null  int64 
 10  이용금액_일시불_R3M        974326 non-null  int64 
 11  이용금액_일시불_R12M       974326 non-null  int64 
 12  이용개월수_신용_R6M        974326 non-null  int64 
 13  이용건수_신용_R6M         974326 non-null  int64 
 14  이용건수_신용_B0M         974326 non-null  int64 
 15  이용건수_신판_R6M         974326 non-null  int64 
 16  이용

In [36]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()
Encoder2 = LabelEncoder()
Encoder3 = LabelEncoder()
Encoder4 = LabelEncoder()
Encoder5 = LabelEncoder()
Encoder6 = LabelEncoder()
Encoder7 = LabelEncoder()
Encoder8 = LabelEncoder()
Encoder9 = LabelEncoder()

Encoder1.fit(all_df['할인건수_R3M'])
Encoder2.fit(all_df['할인건수_B0M'])
Encoder3.fit(all_df['방문횟수_앱_R6M'])
Encoder4.fit(all_df['방문횟수_PC_R6M'])
Encoder5.fit(all_df['인입횟수_ARS_R6M'])
Encoder6.fit(all_df['Life_Stage'])
Encoder7.fit(all_df['이용메뉴건수_ARS_R6M'])
Encoder8.fit(all_df['방문일수_PC_R6M'])
Encoder9.fit(all_df['캠페인접촉건수_R12M'])

LabelEncoder()

In [37]:
all_df['할인건수_R3M'] = Encoder1.transform(all_df['할인건수_R3M'])
all_df['할인건수_B0M'] = Encoder2.transform(all_df['할인건수_B0M'])
all_df['방문횟수_앱_R6M'] = Encoder3.transform(all_df['방문횟수_앱_R6M'])
all_df['방문횟수_PC_R6M'] = Encoder4.transform(all_df['방문횟수_PC_R6M'])
all_df['인입횟수_ARS_R6M'] = Encoder5.transform(all_df['인입횟수_ARS_R6M'])
all_df['Life_Stage'] = Encoder6.transform(all_df['Life_Stage'])
all_df['이용메뉴건수_ARS_R6M'] = Encoder7.transform(all_df['이용메뉴건수_ARS_R6M'])
all_df['방문일수_PC_R6M'] = Encoder8.transform(all_df['방문일수_PC_R6M'])
all_df['캠페인접촉건수_R12M'] = Encoder9.transform(all_df['캠페인접촉건수_R12M'])

In [38]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [39]:
train_df['할인건수_R3M'] = Encoder1.transform(train_df['할인건수_R3M'])
train_df['할인건수_B0M'] = Encoder2.transform(train_df['할인건수_B0M'])
train_df['방문횟수_앱_R6M'] = Encoder3.transform(train_df['방문횟수_앱_R6M'])
train_df['방문횟수_PC_R6M'] = Encoder4.transform(train_df['방문횟수_PC_R6M'])
train_df['인입횟수_ARS_R6M'] = Encoder5.transform(train_df['인입횟수_ARS_R6M'])
train_df['Life_Stage'] = Encoder6.transform(train_df['Life_Stage'])
train_df['이용메뉴건수_ARS_R6M'] = Encoder7.transform(train_df['이용메뉴건수_ARS_R6M'])
train_df['방문일수_PC_R6M'] = Encoder8.transform(train_df['방문일수_PC_R6M'])
train_df['캠페인접촉건수_R12M'] = Encoder9.transform(train_df['캠페인접촉건수_R12M'])

In [53]:
# 라벨 인코더 생성
le = LabelEncoder()

# 문자열 y를 숫자로 변환
train_df['Segment'] = le.fit_transform(train_df['Segment'])

In [54]:
# 입력과 결과로 나눈다.
X = train_df.drop(columns=['Segment','ID','기준년월'])
y = train_df['Segment']

In [55]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-1.43516137, -1.20602485, -1.29214873, ...,  0.42202537,
         0.32015848, -0.8764379 ],
       [-1.43516137, -1.20602485, -1.29214873, ...,  0.42202537,
         0.32015848, -0.8764379 ],
       [-1.43516137, -1.20602485, -1.29214873, ...,  0.42202537,
         0.32015848, -0.8764379 ],
       ...,
       [ 0.8405857 ,  0.03429694,  1.00132125, ..., -1.00545898,
        -0.74095686,  1.08002578],
       [-0.56967601, -0.03952256, -0.4199232 , ..., -0.71996211,
        -0.74095686,  1.08002578],
       [-0.71000197, -0.51207019, -0.56134198, ...,  0.1365285 ,
         0.32015848,  1.08002578]])

In [56]:
scaler_columns = X.columns.tolist()
scaler_columns

['이용금액_R3M_신용체크',
 '_1순위카드이용금액',
 '이용금액_R3M_신용',
 '이용카드수_신용체크',
 'Life_Stage',
 '이용개월수_신용_R12M',
 '이용개월수_신판_R12M',
 '이용개월수_일시불_R12M',
 '이용금액_일시불_R6M',
 '이용금액_일시불_B0M',
 '이용금액_일시불_R3M',
 '이용금액_일시불_R12M',
 '이용개월수_신용_R6M',
 '이용건수_신용_R6M',
 '이용건수_신용_B0M',
 '이용건수_신판_R6M',
 '이용건수_신용_R3M',
 '이용건수_신판_B0M',
 '이용건수_일시불_R6M',
 '이용건수_신판_R3M',
 '이용건수_일시불_B0M',
 '이용개월수_신판_R6M',
 '이용건수_일시불_R3M',
 '이용개월수_일시불_R6M',
 '이용후경과월_신판',
 '이용건수_신용_R12M',
 '이용건수_신판_R12M',
 '이용건수_일시불_R12M',
 '이용가맹점수',
 '이용후경과월_신용',
 '이용금액_오프라인_B0M',
 '이용금액_오프라인_R3M',
 '이용건수_오프라인_B0M',
 '_3순위업종_이용금액',
 '_3순위쇼핑업종_이용금액',
 '_2순위업종_이용금액',
 '이용개월수_오프라인_R6M',
 '이용금액_오프라인_R6M',
 '_2순위쇼핑업종_이용금액',
 '정상청구원금_B5M',
 '정상청구원금_B2M',
 '연속유실적개월수_기본_24M_카드',
 '정상청구원금_B0M',
 '정상입금원금_B0M',
 '정상입금원금_B5M',
 '정상입금원금_B2M',
 '이용개월수_전체_R6M',
 '이용개월수_전체_R3M',
 '청구금액_R6M',
 '청구금액_R3M',
 '청구금액_B0',
 '청구서발송여부_B0',
 '할인건수_R3M',
 '할인건수_B0M',
 '방문횟수_앱_R6M',
 '방문횟수_PC_R6M',
 '방문일수_PC_R6M',
 '인입횟수_ARS_R6M',
 '이용메뉴건수_ARS_R6M',
 '입회경과개월수_신용',
 '_1순위카드이용건수',
 'CA한도금액'

In [57]:
train_X = X2
train_y = y

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [58]:
# LGBM
lgbm_basic_model = LGBMClassifier(verbose=-1)
# 교차 검증을 수행한다
r1 = cross_val_score(lgbm_basic_model, train_X, train_y, scoring='f1', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("LGBM Basic")

print(f'평균 f1 Score : {r1.mean()}')

평균 f1 Score : 0.9036331853639765


In [59]:
# XGBoost
xgboost_basic_model = XGBClassifier(verbose=-1, silent=True)
# 교차 검증을 수행한다
r1 = cross_val_score(xgboost_basic_model, train_X, train_y, scoring='f1', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("XGBoost Basic")

print(f'평균 f1 Score : {r1.mean()}')

평균 f1 Score : 0.9193917115617914


In [60]:
d1 = {
    'f1 score' : f1_score_list
}
result_df = pd.DataFrame(d1, index=model_name_list)
result_df.sort_values(by='f1 score', ascending=False, inplace=True)
result_df

,f1 score
XGBoost Basic,0.919392
LGBM Basic,0.903633


---

In [ ]:
10/0

In [61]:
best_model=xgboost_basic_model.fit(train_X, train_y)

In [63]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(best_model, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(Encoder2, fp)
    pickle.dump(Encoder3, fp)
    pickle.dump(Encoder4, fp)
    pickle.dump(Encoder5, fp)
    pickle.dump(Encoder6, fp)
    pickle.dump(Encoder7, fp)
    pickle.dump(Encoder8, fp)
    pickle.dump(Encoder9, fp)
    pickle.dump(le, fp)

print('저장완료')

저장완료


---
### 혼동행렬